In [4]:
import pygame
import math
import sys

WIDTH, HEIGHT = 800, 450
BALL_RADIUS = 12
POCKET_RADIUS = 20
FRICTION = 0.985
STOP_THRESHOLD = 0.1

WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
GREEN = (34, 139, 34)
BROWN = (139, 69, 19)
RED = (200, 0, 0)
YELLOW = (255, 215, 0)
BLUE = (0, 0, 255)
PURPLE = (128, 0, 128)
ORANGE = (255, 140, 0)

class Ball:
    def __init__(self, x, y, color, is_cue=False):
        self.x = x
        self.y = y
        self.color = color
        self.vx = 0
        self.vy = 0
        self.is_cue = is_cue
        self.active = True

    def move(self):
        if not self.active:
            return
        self.x += self.vx
        self.y += self.vy
        self.vx *= FRICTION
        self.vy *= FRICTION
        if abs(self.vx) < STOP_THRESHOLD:
            self.vx = 0
        if abs(self.vy) < STOP_THRESHOLD:
            self.vy = 0

    def draw(self, screen):
        if self.active:
            pygame.draw.circle(screen, self.color, (int(self.x), int(self.y)), BALL_RADIUS)
            pygame.draw.circle(screen, BLACK, (int(self.x), int(self.y)), BALL_RADIUS, 1)

def check_collision(ball1, ball2):
    dx = ball2.x - ball1.x
    dy = ball2.y - ball1.y
    distance = math.hypot(dx, dy)
    
    if distance < BALL_RADIUS * 2:
        angle = math.atan2(dy, dx)
        force = 5
        
        ball1.vx -= force * math.cos(angle)
        ball1.vy -= force * math.sin(angle)
        ball2.vx += force * math.cos(angle)
        ball2.vy += force * math.sin(angle)
        
        overlap = BALL_RADIUS * 2 - distance
        ball1.x -= overlap * math.cos(angle) / 2
        ball1.y -= overlap * math.sin(angle) / 2
        ball2.x += overlap * math.cos(angle) / 2
        ball2.y += overlap * math.sin(angle) / 2

def main():
    pygame.init()
    screen = pygame.display.set_mode((WIDTH, HEIGHT))
    pygame.display.set_caption("Billiard Game")
    clock = pygame.time.Clock()

    balls = []
    cue_ball = Ball(WIDTH // 4, HEIGHT // 2, WHITE, is_cue=True)
    balls.append(cue_ball)

    colors = [RED, YELLOW, BLUE, PURPLE, ORANGE]
    for i in range(5):
        balls.append(Ball(WIDTH // 2 + i * 25, HEIGHT // 2 - i * 20, colors[i]))
        balls.append(Ball(WIDTH // 2 + i * 25, HEIGHT // 2 + i * 20, colors[i]))

    pockets = [
        (POCKET_RADIUS, POCKET_RADIUS),
        (WIDTH - POCKET_RADIUS, POCKET_RADIUS),
        (POCKET_RADIUS, HEIGHT - POCKET_RADIUS),
        (WIDTH - POCKET_RADIUS, HEIGHT - POCKET_RADIUS),
        (WIDTH // 2, POCKET_RADIUS),
        (WIDTH // 2, HEIGHT - POCKET_RADIUS)
    ]

    running = True
    while running:
        screen.fill(BROWN)
        pygame.draw.rect(screen, GREEN, (20, 20, WIDTH - 40, HEIGHT - 40))

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                mouse_x, mouse_y = pygame.mouse.get_pos()
                if cue_ball.active:
                    dx = mouse_x - cue_ball.x
                    dy = mouse_y - cue_ball.y
                    distance = math.hypot(dx, dy)
                    if distance > 0:
                        cue_ball.vx = (dx / distance) * 8
                        cue_ball.vy = (dy / distance) * 8

        for ball in balls:
            ball.move()

        for i in range(len(balls)):
            for j in range(i + 1, len(balls)):
                if balls[i].active and balls[j].active:
                    check_collision(balls[i], balls[j])

        for ball in balls:
            if not ball.active:
                continue
            if ball.x - BALL_RADIUS < 20 or ball.x + BALL_RADIUS > WIDTH - 20:
                ball.vx *= -1
                ball.x = max(20 + BALL_RADIUS, min(ball.x, WIDTH - 20 - BALL_RADIUS))
            if ball.y - BALL_RADIUS < 20 or ball.y + BALL_RADIUS > HEIGHT - 20:
                ball.vy *= -1
                ball.y = max(20 + BALL_RADIUS, min(ball.y, HEIGHT - 20 - BALL_RADIUS))

        for ball in balls:
            if not ball.active:
                continue
        for px, py in pockets:
            if math.hypot(ball.x - px, ball.y - py) < POCKET_RADIUS:
                    ball.active = False

        for px, py in pockets:
            pygame.draw.circle(screen, BLACK, (px, py), POCKET_RADIUS)

        for ball in balls:
            ball.draw(screen)

        pygame.display.flip()
        clock.tick(60)

    pygame.quit()
    sys.exit()

if __name__ == "__main__":
    main()

SystemExit: 